# pipe_catedra/02_bis — Vecinos + hojas de Random Forest + poda por shadow features

Input  : `z302_features_{modo}.parquet`, `z302_inferencia_{modo}.parquet` (de `02_FE.ipynb`)
Output : `z302bis_features_{modo}.parquet` + `z302bis_inferencia_{modo}.parquet`

Extiende el FE de `02_FE.ipynb` con tres ideas portadas de `otro_pipe`
(ya probadas ahi, ver `otro_pipe/02_Features_hojas_shadow.ipynb`):

1. **Vecinos sustitutos/complementarios**: correlacion (Spearman) entre la
   serie de cada producto y las demas, a nivel PRODUCTO (agregando
   clientes si el modo es `cliente_producto`) -- se agrega el promedio de
   los N vecinos mas correlacionados negativamente (sustitutos) y
   positivamente (complementarios) como features nuevas.
2. **Hojas de Random Forest** (tree embedding): un RF chico entrenado en
   una porcion conservadora de la historia, `.apply()` da el indice de
   hoja por arbol -> features categoricas nuevas.
3. **Poda por shadow features**: se agregan columnas de ruido puro y se
   descarta cualquier feature real (de la base O de las hojas) que no le
   gane a la mejor columna de ruido, en dos competencias separadas (mismo
   fix que en `otro_pipe`: mezclar hojas y features base en la misma
   carrera hace que las hojas -casi un proxy del target- se coman todo el
   gain) con 5 repeticiones y mayoria de votos (una sola corrida de un
   modelo chico es ruidosa).

**Aviso sobre leakage de las hojas/vecinos**: a diferencia de `otro_pipe`
(que tiene un unico split train/val fijo antes del FE), `03_Optuna` de
`pipe_catedra` valida con el esquema `febreros`, que arma VARIOS splits
(uno por cada febrero historico disponible) recien en tiempo de
entrenamiento -- el FE no sabe de antemano cuales van a ser. Por eso el
corte usado aca para calcular vecinos y entrenar el RF (`mes_corte_avanzado`)
es deliberadamente conservador (dos años antes del ultimo periodo
disponible) pero **no es una garantia absoluta** de que ningun split de
`febreros` quede "visto" por el RF/los vecinos -- si te preocupa, subi
`mes_corte_avanzado` a mano para dejarlo antes del primer febrero que
uses en la validacion.

No se toca `03_Optuna.ipynb`/`04_Entrenamiento_final.ipynb`: para usar
este dataset en vez del de `02_FE.ipynb`, solo hay que cambiar
`PARAM['path_input']`/`PARAM['path_train']` a `z302bis_features_*` en esos
notebooks (ambos ya toman TODAS las columnas del parquet como candidatas
a feature, asi que no hace falta ningun otro cambio).


## 0) Setup


In [ ]:
import os, time
from pathlib import Path

import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}")
print(f"salida: {DIR_OUT}")


def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1


def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1


## 1) Parametros — palancas


In [ ]:
PARAM = {
    'experimento': 'z302bis',

    # Debe coincidir con 01_/02_
    'modo_agrupacion': 'producto',
    'solo_predecir': True,
    'horizonte': 2,

    # Vecinos sustitutos/complementarios (a nivel PRODUCTO)
    'n_vecinos': 3,

    # Corte conservador compartido por vecinos y RF: None -> 24 meses antes
    # del ultimo periodo disponible (ver aviso arriba sobre el esquema
    # 'febreros' de 03_Optuna).
    'mes_corte_avanzado': None,
    'meses_atras_default': 24,

    # Hojas de Random Forest
    'n_arboles_rf': 50,
    'profundidad_rf': 6,
    'min_hoja_rf': 50,

    # Poda por shadow features
    'n_shadow': 10,
    'n_repeticiones_shadow': 5,
    'umbral_mayoria_shadow': 0.5,

    'semilla': 102191,

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
}

MODO = PARAM['modo_agrupacion']
_tag_solo = 'solo780' if PARAM['solo_predecir'] else 'todosProductos'
PARAM['path_input']        = str(DIR_OUT / f"z302_features_{MODO}_{_tag_solo}.parquet")
PARAM['path_input_infer']  = str(DIR_OUT / f"z302_inferencia_{MODO}_{_tag_solo}.parquet")
PARAM['path_output']       = str(DIR_OUT / f"z302bis_features_{MODO}_{_tag_solo}.parquet")
PARAM['path_output_infer'] = str(DIR_OUT / f"z302bis_inferencia_{MODO}_{_tag_solo}.parquet")

print('Parametros:', PARAM)


## 2) Carga

Se concatena train + inferencia (con un flag `_es_infer`) porque las
features nuevas (vecinos, hojas de RF) necesitan el panel completo para
calcularse consistentemente en ambos lados -- se separan de nuevo al
guardar.


In [ ]:
df_train_in = pl.read_parquet(PARAM['path_input']).with_columns(pl.lit(False).alias('_es_infer'))
df_infer_in = pl.read_parquet(PARAM['path_input_infer']).with_columns(pl.lit(True).alias('_es_infer'))
df = pl.concat([df_train_in, df_infer_in], how='diagonal_relaxed')

ULTIMO_PERIODO = int(df['periodo'].max())
print(f'Train: {df_train_in.shape}   Inferencia: {df_infer_in.shape}   Combinado: {df.shape}')
print(f'Ultimo periodo disponible: {ULTIMO_PERIODO}')

if PARAM['mes_corte_avanzado'] is None:
    PARAM['mes_corte_avanzado'] = desplazar_meses(ULTIMO_PERIODO, -PARAM['meses_atras_default'])
    print(f"mes_corte_avanzado no seteado -> uso {PARAM['mes_corte_avanzado']} "
         f"({PARAM['meses_atras_default']} meses antes del ultimo periodo)")

MES_CORTE = PARAM['mes_corte_avanzado']
print(f'Corte para vecinos y RF: periodo < {MES_CORTE}')


## 3) Vecinos sustitutos/complementarios (a nivel PRODUCTO)

Se agrega `tn` a nivel `product_id x periodo` (en modo `producto` es
identidad; en `cliente_producto` suma todos los clientes de ese producto).
La correlacion se calcula SOLO con periodos anteriores a `mes_corte_avanzado`
(ver aviso de leakage arriba). Para cada producto: los `n_vecinos` con
correlacion mas NEGATIVA (sustitutos -- suben cuando este baja) y los
`n_vecinos` con correlacion mas POSITIVA (complementarios).


In [ ]:
t0 = time.time()

tot_prod = (df.group_by(['product_id', 'periodo'])
              .agg(pl.col('tn').sum().alias('tn_prod')))

wide = (tot_prod.filter(pl.col('periodo') < MES_CORTE)
                .pivot(on='product_id', index='periodo', values='tn_prod')
                .sort('periodo')
                .drop('periodo'))
corr = wide.to_pandas().corr(method='spearman')
print(f'Matriz de correlacion: {corr.shape}   [{time.time()-t0:.1f}s]')

N_VEC = PARAM['n_vecinos']
vecinos_rows = []
for p in corr.columns:
    s = corr[p].drop(labels=[p]).dropna()
    if s.empty:
        continue
    for vecino, r in s.sort_values().head(N_VEC).items():
        vecinos_rows.append({'product_id': p, 'tipo': 'sustituto', 'vecino_id': vecino, 'corr': float(r)})
    for vecino, r in s.sort_values(ascending=False).head(N_VEC).items():
        vecinos_rows.append({'product_id': p, 'tipo': 'complementario', 'vecino_id': vecino, 'corr': float(r)})

vecinos_pd = pd.DataFrame(vecinos_rows)
vecinos = pl.from_pandas(vecinos_pd).with_columns([
    pl.col('product_id').cast(pl.Int32), pl.col('vecino_id').cast(pl.Int32)
])
print(f'Vecinos calculados para {vecinos["product_id"].n_unique()} productos')

feat_vecinos = (vecinos.join(tot_prod.rename({'product_id': 'vecino_id', 'tn_prod': 'tn_vecino'}),
                             on='vecino_id', how='left')
                       .group_by(['product_id', 'tipo', 'periodo'])
                       .agg(pl.col('tn_vecino').mean().alias('tn_vecino_prom')))

feat_vecinos_piv = (feat_vecinos.collect() if isinstance(feat_vecinos, pl.LazyFrame) else feat_vecinos) \
    .pivot(on='tipo', index=['product_id', 'periodo'], values='tn_vecino_prom')

for col_falt in ('sustituto', 'complementario'):
    if col_falt not in feat_vecinos_piv.columns:
        feat_vecinos_piv = feat_vecinos_piv.with_columns(pl.lit(None, dtype=pl.Float64).alias(col_falt))

feat_vecinos_piv = feat_vecinos_piv.rename({
    'sustituto': 'tn_sustitutos_prom', 'complementario': 'tn_complementarios_prom'
})

df = (df.join(feat_vecinos_piv, on=['product_id', 'periodo'], how='left')
        .with_columns([
            pl.col('tn_sustitutos_prom').fill_null(0.0),
            pl.col('tn_complementarios_prom').fill_null(0.0),
        ]))
print(f'tn_sustitutos_prom / tn_complementarios_prom agregados.   [{time.time()-t0:.1f}s]')


## 4) Hojas de Random Forest (tree embedding)

Un Random Forest chico (arboles poco profundos, hoja minima alta para no
explotar la cardinalidad categorica) entrenado SOLO con filas de
`periodo < mes_corte_avanzado`. El indice de hoja donde cae cada fila, por
arbol, se agrega como columna categorica nueva -- se aplica a TODAS las
filas (train e inferencia), pero el AJUSTE del RF solo usa la porcion
vieja de la historia.


In [ ]:
t0 = time.time()

COLS_ID = ['agrupa_id', 'product_id', 'customer_id', 'periodo', '_es_infer']
COLS_TARGET = ['tn', 'tn_t2', 'target_nivel', 'target_delta']
COLS_META = ['modo_agrupacion', 'solo_predecir']
PROHIBIDAS_BASE = set(COLS_ID) | set(COLS_TARGET) | set(COLS_META)

FEATURES_BASE = [c for c in df.columns if c not in PROHIBIDAS_BASE]
CAT_BASE = [c for c in PARAM['cols_categoricas'] if c in FEATURES_BASE]
NUM_RF = [c for c in FEATURES_BASE if c not in CAT_BASE]
print(f'FEATURES base (antes de hojas/shadow): {len(FEATURES_BASE)} ({len(NUM_RF)} numericas, {len(CAT_BASE)} categoricas)')

PARAM_RF = dict(n_estimators=PARAM['n_arboles_rf'], max_depth=PARAM['profundidad_rf'],
               min_samples_leaf=PARAM['min_hoja_rf'], n_jobs=-1, random_state=PARAM['semilla'])

_tr_rf = df.filter((pl.col('periodo') < MES_CORTE) & pl.col('target_nivel').is_not_null())
X_rf_train = _tr_rf.select(NUM_RF).fill_null(0.0).to_numpy()
y_rf_train = _tr_rf['target_nivel'].fill_null(0.0).to_numpy()

rf = RandomForestRegressor(**PARAM_RF)
rf.fit(X_rf_train, y_rf_train)
print(f'Random Forest entrenado: {PARAM_RF["n_estimators"]} arboles x {len(NUM_RF)} features numericas, '
     f'{X_rf_train.shape[0]:,} filas (periodo < {MES_CORTE}).   [{time.time()-t0:.1f}s]')

X_todo = df.select(NUM_RF).fill_null(0.0).to_numpy()
hojas = rf.apply(X_todo)
COLS_HOJA = [f'hoja_arbol_{i}' for i in range(hojas.shape[1])]
df = df.with_columns([pl.Series(c, hojas[:, i].astype(str)) for i, c in enumerate(COLS_HOJA)])
print(f'{len(COLS_HOJA)} columnas de hoja agregadas (categoricas, una por arbol).   [{time.time()-t0:.1f}s]')


## 5) Poda por shadow features — DOS rondas, repetidas, mayoria de votos

Mismo fix que en `otro_pipe`: features base y hojas de RF compiten en
carreras SEPARADAS (las hojas son casi un proxy del target y se comen el
gain si compiten junto a las features base), con un modelo chico y
regularizado (para no sobreajustar en pocas filas) y 5 repeticiones con
columnas de ruido distintas cada vez -- sobrevive lo que le gana al ruido
en la MAYORIA de las repeticiones.


In [ ]:
t0 = time.time()
rng = np.random.default_rng(PARAM['semilla'])
N_SHADOW = PARAM['n_shadow']
N_REP = PARAM['n_repeticiones_shadow']
UMBRAL = PARAM['umbral_mayoria_shadow']

_sup = df.filter(pl.col('target_nivel').is_not_null())
_periodos_sup = sorted(_sup['periodo'].unique().to_list())
# Split simple para el diagnostico de shadow: los ultimos 2 periodos de
# TRAIN como validacion interna (no tiene que ver con el esquema de
# 03_Optuna, es solo para medir importancia).
_val_periodos = set(_periodos_sup[-2:])
_tr_shadow = _sup.filter(~pl.col('periodo').is_in(_val_periodos))
_va_shadow = _sup.filter(pl.col('periodo').is_in(_val_periodos))
_n_tr_shadow = _tr_shadow.height
print(f'shadow: train {_tr_shadow.height:,} filas, val {_va_shadow.height:,} filas')


def _cols_ruido(n, prefijo):
    return [pl.Series(f'{prefijo}_{i}', rng.normal(size=n) if i % 2 == 0 else rng.uniform(-1, 1, size=n))
           for i in range(N_SHADOW)]


def _sobreviven_por_shadow(features_candidatas, cats_candidatas, prefijo_shadow):
    conteos = {c: 0 for c in features_candidatas}
    _min_hoja = max(20, _n_tr_shadow // 40)
    for rep in range(N_REP):
        cols_shadow = [f'{prefijo_shadow}{rep}_{i}' for i in range(N_SHADOW)]
        tr = _tr_shadow.with_columns(_cols_ruido(_tr_shadow.height, f'{prefijo_shadow}{rep}'))
        va = _va_shadow.with_columns(_cols_ruido(_va_shadow.height, f'{prefijo_shadow}{rep}'))

        pool = features_candidatas + cols_shadow
        cats_pool = [c for c in cats_candidatas if c in pool]

        def _a_pandas_pool(df_pl):
            out = (df_pl.select(pool + ['target_nivel'])
                        .with_columns([pl.col(c).cast(pl.Utf8).cast(pl.Categorical) for c in cats_pool])
                        .to_pandas())
            for c in cats_pool:
                out[c] = out[c].astype('category')
            return out

        df_pd_tr, df_pd_va = _a_pandas_pool(tr), _a_pandas_pool(va)
        for c in cats_pool:
            df_pd_va[c] = df_pd_va[c].cat.set_categories(df_pd_tr[c].cat.categories)

        modelo = lgb.LGBMRegressor(objective='regression', metric='mae', verbosity=-1,
                                   n_estimators=100, learning_rate=0.05, num_leaves=15,
                                   min_child_samples=_min_hoja, reg_alpha=1.0, reg_lambda=1.0,
                                   seed=PARAM['semilla'] + rep, n_jobs=-1)
        modelo.fit(df_pd_tr[pool], df_pd_tr['target_nivel'], categorical_feature=cats_pool,
                  eval_set=[(df_pd_va[pool], df_pd_va['target_nivel'])],
                  callbacks=[lgb.early_stopping(20, verbose=False)])

        imp = pd.Series(modelo.booster_.feature_importances_ if hasattr(modelo.booster_, 'feature_importances_')
                        else modelo.booster_.feature_importance(importance_type='gain'), index=pool)
        piso = float(imp[cols_shadow].max())
        for c in features_candidatas:
            if imp.get(c, 0.0) > piso:
                conteos[c] += 1

    sobreviven = [c for c in features_candidatas if conteos[c] / N_REP > UMBRAL]
    return sobreviven, conteos


FEATURES_SOBREVIVEN_BASE, _conteos_base = _sobreviven_por_shadow(FEATURES_BASE, CAT_BASE, '_shadowA')
_descartadas_base = [c for c in FEATURES_BASE if c not in FEATURES_SOBREVIVEN_BASE]
print(f'[ronda 1: features base] candidatas: {len(FEATURES_BASE)}   sobreviven: {len(FEATURES_SOBREVIVEN_BASE)}')
if _descartadas_base:
    print(f'  descartadas: {_descartadas_base}')

FEATURES_SOBREVIVEN_HOJA, _conteos_hoja = _sobreviven_por_shadow(COLS_HOJA, COLS_HOJA, '_shadowB')
print(f'[ronda 2: hojas de RF] candidatas: {len(COLS_HOJA)}   sobreviven: {len(FEATURES_SOBREVIVEN_HOJA)}')

FEATURES_FINALES = FEATURES_SOBREVIVEN_BASE + FEATURES_SOBREVIVEN_HOJA
if not FEATURES_FINALES:
    raise RuntimeError('Ninguna feature sobrevivio la poda por shadow -- revisar N_REP/UMBRAL.')
print(f'\nFEATURES finales: {len(FEATURES_FINALES)} (de {len(FEATURES_BASE) + len(COLS_HOJA)} candidatas)   '
     f'[{time.time()-t0:.1f}s]')


## 6) Armar output podado + split train/inferencia


In [ ]:
COLS_SIEMPRE = COLS_ID + COLS_TARGET + COLS_META
COLS_SIEMPRE = [c for c in COLS_SIEMPRE if c in df.columns]

df_final = df.select(COLS_SIEMPRE + FEATURES_FINALES)

df_train_out = df_final.filter(~pl.col('_es_infer')).drop('_es_infer')
df_infer_out = df_final.filter(pl.col('_es_infer')).drop('_es_infer')

print(f'Train: {df_train_out.shape}   Inferencia: {df_infer_out.shape}')

df_train_out.write_parquet(PARAM['path_output'])
print(f'Guardado: {PARAM["path_output"]}')

df_infer_out.write_parquet(PARAM['path_output_infer'])
print(f'Guardado: {PARAM["path_output_infer"]}')

print(f'\nPara usar este dataset en 03_Optuna.ipynb / 04_Entrenamiento_final.ipynb, cambia:')
print(f"  PARAM['path_input']/PARAM['path_train'] -> z302bis_features_{MODO}.parquet")
print(f"  PARAM['path_infer'] (04_) -> z302bis_inferencia_{MODO}.parquet")
